# Análisis exploratorio (EDA) del dataset TUSZ v2.0.6

Segundo paso de la **Fase 1 (exploración de datos)**. Analiza el inventario construido en `01_inventario_tusz.ipynb` para responder a las preguntas que condicionan el resto del TFM:

- **Fase 2 (preprocesado)**: ¿qué montajes y frecuencias de muestreo hay que homogeneizar?
- **Fase 3 (predicción)**: ¿cuántas crisis tienen suficiente lead-in para recortar una ventana preictal? ¿Qué umbral usamos?
- **Fase 4 (clasificación)**: ¿cuántas crisis y pacientes hay de cada tipo?

No lee el dataset: trabaja con `files.csv` y `seizures.csv`, así que se ejecuta en segundos. **Requisito**: haber ejecutado antes `01_inventario_tusz.ipynb`.

## Conceptos y convención visual

- **Lead-in**: tiempo sin crisis que hay antes de cada crisis, dentro de su mismo fichero. La definición completa, con ejemplo, está al inicio de `01_inventario_tusz.ipynb`.
- **Ventana preictal**: trozo de señal antes de la crisis que el modelo aprenderá a reconocer. La elegimos nosotros y tiene que caber dentro del lead-in.

**Iconos** (los mismos en todo el TFM, definidos en `tfm_comun.py`):

| Etiquetas | | Particiones | |
|---|---|---|---|
| ⬜ `bckg` fondo | 🟩 `absz` ausencia | 🔵 `train` | entrenamiento |
| ⬛ `seiz` / `sin_tipo` crisis sin tipo | 🟪 `tcsz` tónico-clónica | 🟡 `dev` | validación |
| 🟥 `fnsz` focal no específica | 🟨 `tnsz` tónica | 🟢 `eval` | test final |
| 🟦 `gnsz` generalizada no específica | 🟫 `spsz` parcial simple | ⚪ todas | dataset completo |
| 🟧 `cpsz` parcial compleja | 🔳 `mysz` mioclónica | 🔴 | avisos |

En las figuras se usan colores a juego: azul 🔵, amarillo 🟡 y verde 🟢.

## 1. Importación de librerías

Análisis con `pandas` y figuras con `matplotlib`. La convención visual se importa de `tfm_comun.py`.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tfm_comun import SPLITS, ICONO_ETIQUETA, ICONO_SPLIT, COLORES_SPLIT, con_icono, tabla_con_iconos

## 2. Configuración y carga del inventario

- `RUTA_INVENTARIO`: salidas de `01_inventario_tusz.ipynb`.
- `RUTA_RESULTADOS`: `outputs/02_eda`, con sus `figuras/`.
- `UMBRALES_LEADIN_MIN`: lead-in mínimos (en minutos) con los que se mide cuántas crisis sirven para predicción.
- `LEADIN_ELEGIDO_MIN`: umbral elegido para el TFM (5 min). La justificación está en la sección 5.

Si no existen los CSV del inventario, la celda avisa de que hay que ejecutar antes el notebook 01.

In [ ]:
# Rutas
RUTA_INVENTARIO = Path('./outputs/01_inventario')
RUTA_RESULTADOS = Path('./outputs/02_eda')
RUTA_FIGURAS = RUTA_RESULTADOS / 'figuras'
for carpeta in [RUTA_RESULTADOS, RUTA_FIGURAS]:
    carpeta.mkdir(parents=True, exist_ok=True)

# Parámetros del análisis
UMBRALES_LEADIN_MIN = [2, 5, 7, 10, 15]
LEADIN_ELEGIDO_MIN = 5

# Carga del inventario
for nombre in ['files.csv', 'seizures.csv', 'configuracion.json']:
    assert (RUTA_INVENTARIO / nombre).exists(), \
        f'No encuentro {RUTA_INVENTARIO / nombre}. Ejecuta antes 01_inventario_tusz.ipynb.'

df_ficheros = pd.read_csv(RUTA_INVENTARIO / 'files.csv')
df_crisis = pd.read_csv(RUTA_INVENTARIO / 'seizures.csv')
with open(RUTA_INVENTARIO / 'configuracion.json', encoding='utf-8') as f:
    config_inventario = json.load(f)

print(f'Inventario generado el {config_inventario["fecha"]} a partir de {config_inventario["ruta_tusz"]}')
print(f'{len(df_ficheros)} ficheros y {len(df_crisis)} crisis cargados')

config = {
    'inventario': str(RUTA_INVENTARIO),
    'umbrales_leadin_min': UMBRALES_LEADIN_MIN,
    'leadin_elegido_min': LEADIN_ELEGIDO_MIN,
    'fecha': time.strftime('%Y-%m-%d %H:%M'),
}
with open(RUTA_RESULTADOS / 'configuracion.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

## 3. Panorama general por partición

Cuántos pacientes, sesiones, ficheros, horas de registro y crisis hay en cada partición. También comprobamos que **ningún paciente aparece en dos particiones**: si apareciera, el modelo vería en test datos de un paciente con el que ha entrenado y los resultados serían optimistas (*data leakage*).

In [ ]:
# Tabla 1: resumen por split
filas = []
for s in SPLITS:
    d = df_ficheros[df_ficheros.split == s]
    filas.append({
        'split': s,
        'pacientes': d.patient.nunique(),
        'sesiones': d[['patient', 'session']].drop_duplicates().shape[0],
        'ficheros': len(d),
        'horas': round(d.edf_dur_s.sum() / 3600, 1),
        'pac. con crisis': d[d.n_seiz_events_bi > 0].patient.nunique(),
        'crisis': d.n_seiz_events_bi.sum(),
        'min. en crisis': round(d.seiz_secs_bi.sum() / 60, 1),
    })
tabla_splits = pd.DataFrame(filas).set_index('split')
pacientes = {s: set(df_ficheros[df_ficheros.split == s].patient) for s in SPLITS}
compartidos = {
    'train/dev': len(pacientes['train'] & pacientes['dev']),
    'train/eval': len(pacientes['train'] & pacientes['eval']),
    'dev/eval': len(pacientes['dev'] & pacientes['eval']),
}
print('Pacientes compartidos entre particiones (debe ser 0):', compartidos)
tabla_con_iconos(tabla_splits)

## 4. Tipos de crisis

Eventos, pacientes y minutos de cada tipo en cada partición. Es la tabla clave para la **Fase 4 (clasificación)**.

Cada crisis es un evento del `.csv_bi`; su tipo es la etiqueta dominante del `.csv` (ver sección 6 del notebook 01).

En la figura, la escala es logarítmica porque 🟥 `fnsz` tiene cientos de veces más eventos que 🔳 `mysz`. El número de **pacientes** importa más que el de crisis: un tipo con muchas crisis pero de un solo paciente no permite generalizar.

In [ ]:
# Tabla 2: tipos de crisis por split
df_crisis['dur_s'] = df_crisis.stop - df_crisis.start
agrupado = df_crisis.groupby(['label', 'split'])
tabla_tipos = pd.concat({
    'crisis': agrupado.size().unstack(),
    'pacientes': agrupado.patient.nunique().unstack(),
    'minutos': (agrupado.dur_s.sum() / 60).round(1).unstack(),
}, axis=1)
columnas = pd.MultiIndex.from_product([['crisis', 'pacientes', 'minutos'], SPLITS])
tabla_tipos = tabla_tipos.reindex(columns=columnas, fill_value=0).fillna(0)
for metrica in ['crisis', 'pacientes']:
    tabla_tipos[metrica] = tabla_tipos[metrica].astype(int)
tabla_tipos.index.name = 'tipo'
tabla_con_iconos(tabla_tipos)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (metrica, titulo) in zip(axes, [('crisis', 'Número de crisis por tipo'),
                                         ('pacientes', 'Número de pacientes por tipo')]):
    tabla_tipos[metrica].plot.bar(ax=ax, color=[COLORES_SPLIT[s] for s in SPLITS], width=0.8)
    ax.set_yscale('log')
    ax.set_title(titulo)
    ax.set_xlabel('Tipo de crisis')
    ax.set_ylabel(f'{metrica.capitalize()} (escala log)')
    ax.grid(True, axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / 'crisis_y_pacientes_por_tipo.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Lead-in y elección del umbral para predicción

Para la **Fase 3 (predicción)** solo sirven las crisis con suficiente lead-in para que quepa una ventana preictal.

### 5.1. Elección del umbral de lead-in

No fijamos el umbral por costumbre (en la literatura se usan 30–60 min con registros de días, como CHB-MIT). Lo elegimos con los datos de TUSZ, mirando tres cosas:

1. **Cuánto dura un fichero.** Como mucho 30 min y la mediana es 10 min, así que un lead-in de 30 min es imposible dentro de un fichero.
2. **Cada cuánto llegan las crisis.** Muchas van en racimos: si la crisis anterior acaba de terminar, la "ventana preictal" sería en realidad el final de esa crisis. El umbral sirve para quedarnos con crisis precedidas de señal limpia.
3. **Cuántos pacientes quedan en eval.** Es lo que manda: con pocos pacientes en eval, cualquier métrica mide a esos pocos pacientes y no generaliza.

El umbral elegido es el último antes de que eval se quede con muy pocos pacientes.

In [ ]:
# Intervalo entre crisis consecutivas del mismo fichero (excluye la primera de cada fichero)
intervalos_min = df_crisis.loc[~df_crisis.primera_del_fichero, 'leadin_s'] / 60
print('Intervalo entre crisis consecutivas (min): '
      + ', '.join(f'p{int(q * 100)}={v:.1f}' for q, v in intervalos_min.quantile([0.25, 0.5, 0.75, 0.9]).items()))
print(f'Duración de fichero: mediana {df_ficheros.edf_dur_s.median() / 60:.0f} min, máximo {df_ficheros.edf_dur_s.max() / 60:.0f} min\n')

# Crisis y pacientes que quedan con cada umbral
filas = []
for umbral in UMBRALES_LEADIN_MIN:
    fila = {'lead-in mínimo (min)': umbral}
    for s in SPLITS:
        validas = df_crisis[(df_crisis.split == s) & (df_crisis.leadin_s >= umbral * 60)]
        fila[f'{s} crisis'] = len(validas)
        fila[f'{s} pacientes'] = validas.patient.nunique()
    filas.append(fila)
tabla_umbrales = pd.DataFrame(filas).set_index('lead-in mínimo (min)')
tabla_umbrales.to_csv(RUTA_RESULTADOS / 'crisis_por_umbral_leadin.csv')
tabla_con_iconos(tabla_umbrales)

In [ ]:
# Curva: pacientes de cada split que conservan al menos una crisis válida según el umbral
umbrales_finos = np.arange(0.5, 15.5, 0.5)
fig, ax = plt.subplots(figsize=(9, 4.5))
for s in SPLITS:
    d = df_crisis[df_crisis.split == s]
    ax.plot(umbrales_finos, [d[d.leadin_s >= u * 60].patient.nunique() for u in umbrales_finos],
            marker='o', markersize=3, label=s, color=COLORES_SPLIT[s])
ax.axvline(LEADIN_ELEGIDO_MIN, color='red', linestyle='--', label=f'Umbral elegido ({LEADIN_ELEGIDO_MIN} min)')
ax.set_xlabel('Lead-in mínimo (min)')
ax.set_ylabel('Pacientes con al menos una crisis válida')
ax.set_title('Pacientes disponibles para predicción según el lead-in exigido')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / 'pacientes_segun_leadin.png', dpi=150, bbox_inches='tight')
plt.show()

elegidas = df_crisis[df_crisis.leadin_s >= LEADIN_ELEGIDO_MIN * 60]
print(f'Con lead-in ≥ {LEADIN_ELEGIDO_MIN} min: ' + ' | '.join(
    f'{con_icono(s)}: {(elegidas.split == s).sum()} crisis de {elegidas[elegidas.split == s].patient.nunique()} pacientes'
    for s in SPLITS))

### 5.2. Viabilidad para predicción por partición

Cuántas crisis (y de cuántos pacientes) superan cada umbral, y cómo se reparte el lead-in de todas las crisis.

In [ ]:
# Tabla 3: viabilidad para predicción
filas = []
for s in SPLITS:
    d = df_crisis[df_crisis.split == s]
    fila = {'split': s, 'crisis': len(d)}
    for umbral in UMBRALES_LEADIN_MIN:
        validas = d[d.leadin_s >= umbral * 60]
        fila[f'lead-in ≥ {umbral} min'] = len(validas)
        fila[f'pacientes ({umbral} min)'] = validas.patient.nunique()
    filas.append(fila)
tabla_prediccion = pd.DataFrame(filas).set_index('split')
tabla_con_iconos(tabla_prediccion)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
leadin_min = df_crisis.leadin_s / 60
bins = np.logspace(np.log10(max(leadin_min.min(), 0.01)), np.log10(leadin_min.max()), 40)
for s in SPLITS:
    ax.hist(leadin_min[df_crisis.split == s], bins=bins, alpha=0.6, label=s, color=COLORES_SPLIT[s])
ax.axvline(LEADIN_ELEGIDO_MIN, color='red', linestyle='--', label=f'Umbral elegido ({LEADIN_ELEGIDO_MIN} min)')
ax.set_xscale('log')
ax.set_xlabel('Lead-in (minutos, escala log)')
ax.set_ylabel('Número de crisis')
ax.set_title('Tiempo sin crisis antes de cada crisis (dentro del fichero)')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / 'distribucion_leadin.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Heterogeneidad técnica: montajes y frecuencias de muestreo

Los ficheros no se grabaron todos igual. Esta figura muestra lo que habrá que homogeneizar en la **Fase 2 (preprocesado)**: remuestrear a una frecuencia común y construir el montaje TCP de 22 canales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pd.crosstab(df_ficheros.montage, df_ficheros.split).reindex(columns=SPLITS, fill_value=0).plot.bar(
    ax=axes[0], color=[COLORES_SPLIT[s] for s in SPLITS])
axes[0].set_title('Ficheros por montaje')
axes[0].set_xlabel('Montaje')
pd.crosstab(df_ficheros.fs.round().astype(int), df_ficheros.split).reindex(columns=SPLITS, fill_value=0).plot.bar(
    ax=axes[1], color=[COLORES_SPLIT[s] for s in SPLITS])
axes[1].set_title('Ficheros por frecuencia de muestreo')
axes[1].set_xlabel('Hz')
for ax in axes:
    ax.set_ylabel('Número de ficheros')
    ax.grid(True, axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=0)
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / 'montajes_y_frecuencias.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Resumen → `summary.txt`

Reúne en un fichero de texto las tablas de las secciones 3, 4 y 5, junto con montajes, frecuencias y duraciones. Sirve para consultar los números sin abrir el notebook.

In [ ]:
duraciones = df_ficheros.edf_dur_s
texto = '\n\n'.join([
    '=== EDA TUSZ ===\n'
    f'Dataset: {config_inventario["ruta_tusz"]}\n'
    f'Inventario: {config_inventario["fecha"]} | EDA: {config["fecha"]}',
    '--- Resumen por split ---\n' + tabla_con_iconos(tabla_splits).to_string(),
    f'Pacientes compartidos entre splits (debe ser 0): {compartidos}',
    f'Montajes: {dict(Counter(df_ficheros.montage))}\n'
    f'Frecuencias de muestreo (Hz): {dict(Counter(df_ficheros.fs.round().astype(int)))}\n'
    f'Nº de señales por EDF: {dict(sorted(Counter(df_ficheros.n_signals).items()))}\n'
    f'Duración de fichero min/mediana/max (s): '
    f'{duraciones.min():.0f} / {duraciones.median():.0f} / {duraciones.max():.0f}',
    '--- Tipos de crisis (1 crisis = 1 evento del .csv_bi; tipo = etiqueta dominante del .csv) ---\n'
    + tabla_con_iconos(tabla_tipos).to_string(),
    '--- Viabilidad para predicción (lead-in = tiempo sin crisis previo dentro del mismo fichero) ---\n'
    f'Umbral elegido: {LEADIN_ELEGIDO_MIN} min\n'
    + tabla_con_iconos(tabla_prediccion).to_string(),
]) + '\n'

(RUTA_RESULTADOS / 'summary.txt').write_text(texto, encoding='utf-8')
print(texto)

## 8. Conclusiones para el TFM

A revisar con los resultados de las celdas anteriores:

- **Splits limpios**: ningún paciente compartido entre 🔵 `train`, 🟡 `dev` y 🟢 `eval`.
- **Heterogeneidad técnica**: 3 montajes y 5 frecuencias de muestreo (mayoría a 256 Hz). En la Fase 2 habrá que remuestrear a una frecuencia común y construir el montaje TCP de 22 canales.
- **Desequilibrio de tipos**: 🟥 `fnsz` y 🟦 `gnsz` dominan. Las ausencias 🟩 `absz` tienen pocas crisis y, en 🟢 `eval`, todas son de un único paciente: cualquier métrica de 🟩 `absz` en 🟢 `eval` mide en realidad a ese paciente.
- **Lead-in solo dentro del fichero**: no se puede comprobar que los tokens de una sesión sean seguidos, así que no se encadenan.
- **Umbral de 5 min**: es el mayor que conserva un número razonable de pacientes en eval. Permite, por ejemplo, una ventana preictal de 4 min más 1 min de margen antes de la crisis.
- **Enfoque**: con TUSZ no es viable la predicción clásica a 30–60 min. Lo realista es la **anticipación a corto plazo** (minutos antes de la crisis). Conviene explicarlo así en la memoria.

## 9. Archivos generados

En `outputs/02_eda/`:

- `configuracion.json`: parámetros del análisis y fecha.
- `crisis_por_umbral_leadin.csv`: crisis y pacientes por partición para cada umbral de lead-in.
- `summary.txt`: tablas resumen (con iconos).
- `figuras/`: `crisis_y_pacientes_por_tipo.png`, `pacientes_segun_leadin.png`, `distribucion_leadin.png`, `montajes_y_frecuencias.png`.